# Data Preparation



Aufbereitung des Master-Datensatzes für die Modellierung.

| Phase | Was passiert | Wann | Leakage-Risiko |
|:---|:---|:---|:---|
| **Cleaning** | Fehler auf Basis von Domainwissen entfernen | Vor Split | Keins |
| **Split** | Temporal aufteilen — kein Shuffle | — | Keins |
| **Preprocessing** | Lücken füllen, Werte anpassen | Nach Split | Nur aus Train |
| **Feature Engineering** | Neue Spalten ableiten | Nach Split | Encodings aus Train |

> **Leakage-Prinzip:** Alles was Parameter aus den Daten lernt kommt nach dem Split —  
> gefittet auf Train, angewendet auf Test.

## Agenda



Aus `01_exploration.ipynb` — diese Tabelle bestimmt was in welcher Phase bereinigt wird.

| Topic | Befund | Empfehlung | Phase |
|:---|:---|:---|:---:|
| **Delay** | Extreme Werte ±8.3h — physikalisch nicht plausibel | Rausfiltern `\|delay\| > 3.600s` | 1 |
| **Delay** | 74.669 Zeilen: Schedule vorhanden, aber kein Delay | Herausfiltern | 1 |
| **BPUIC** | 0.10% anomale IDs > 100.000.000 | Rausfiltern | 1 |
| **Meteo** | `humidity` > 100% — Sensor-Kalibrierungsdrift | `.clip(0, 100)` | 1 |
| **Datenqualität** | 1.72% Duplikate | `distinct()` | 1 |
| **Events** | 78.5% null — Normalfall kein Event | `null` → `"kein_event"` | 1 |
| **District** | 6.87% null — Haltestellen außerhalb Stadtgebiet | `null` → `"ausserhalb"` | 1 |
| **Meteo** | Stündliche Messausfälle (~0.14–0.35%), zeitlich klumpend | Forward/Backward Fill ±2h | 3 |
| **Meteo** | `precipitation` zero-inflated | Flag `hat_regen` + Wert | 4 |
| **Delay** | Skewness 38–43 — non-linear, long tail | Keine Bereinigung — XGBoost | — |
| **Meteo** | Wetter→Delay: r max 0.03 linear | Schwellenwert-Flags als Features | 4 |

> **Prognose:** ~1,7 Mio. Zeilen (~2%) werden durch Phase 1 entfernt.  
> Nach Phase 1 verbleiben **~86–87 Mio. Zeilen**.

## Setup



### Imports

In [ ]:
import polars as pl
from pathlib import Path

from wgnd.core.theme import setup
from wgnd.core._output import section_header, log, success, warn

from zh_tram_flow.config import PATHS, RANDOM_SEED
from zh_tram_flow.cleaning import (
    structural_cleaning_pipeline,
    impute_meteo_lazy,
    report_step,
    METEO_COLS,
)

setup()

%load_ext autoreload
%autoreload 2

### Pfade und Datensätze

Pfade und Konfiguration kommen aus `zh_tram_flow.config`.  
Der Master-Datensatz wird als LazyFrame geladen — kein RAM-Verbrauch bis zur ersten Operation.  
Für eine bessere Handhabung der großen Datenmenge werden zusätzlich Jahres-Pakete erstellt.

In [ ]:
DATA      = PATHS["raw"]       / "zh-tram-data-master.parquet"
INTERIM   = PATHS["interim"]
PROCESSED = PATHS["processed"]

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# Gesamte Datei

# Lazy Scan — kein RAM-Verbrauch bis collect()
lf_raw = pl.scan_parquet(DATA)

n_raw    = lf_raw.select(pl.len()).collect().item()
date_min = lf_raw.select(pl.col("operating_date").min()).collect().item()
date_max = lf_raw.select(pl.col("operating_date").max()).collect().item()

log(f"\n{DATA}")
log(f"Datensatz: {n_raw:,} Zeilen · {len(lf_raw.collect_schema())} Spalten")
log(f"Zeitraum:  {date_min} bis {date_max}")


# Aufteilung in Jahres-Dateien

years_to_split = ['2023', '2024', '2025']
lazyframes = {'raw': lf_raw}

for year in years_to_split:
    # save file
    file_path = (INTERIM / f"zh-tram-data-{year}.parquet")
    lf_raw.filter(pl.col("operating_date").dt.year() == int(year)).sink_parquet(file_path)
    # load file
    lazyframes[year] = pl.scan_parquet(file_path)
    n_raw    = lazyframes[year].select(pl.len()).collect().item()
    date_min = lazyframes[year].select(pl.col("operating_date").min()).collect().item()
    date_max = lazyframes[year].select(pl.col("operating_date").max()).collect().item()
    # output
    log(f"\n{file_path}")
    log(f"Datensatz: {n_raw:,} Zeilen · {len(lf_raw.collect_schema())} Spalten")
    log(f"Zeitraum:  {date_min} bis {date_max}")



## Cleaning


*Fehler entfernen auf Basis von Domainwissen — vor dem Split, kein Leakage-Risiko.*

Nur Regeln die unabhängig von Statistiken gelten:

| Schritt | Was | Warum |
|:---|:---|:---|
| Duplikate | Vollständig doppelte Zeilen entfernen | GTFS-Join-Artefakt |
| BPUIC | Anomale Haltestellen-IDs entfernen | Außerhalb VBZ-Bereich |
| Delay-Mismatch | Schedule ohne Delay entfernen | Übertragungsfehler |
| Extreme Delays | `\|delay\| > 3.600s` entfernen | Physikalisch nicht plausibel |
| Humidity | Über 100% kappen | Sensor-Kalibrierungsdrift |
| Null-Kategorien | District und Events mit Label füllen | Kein Fehler — definierter Zustand |

In [ ]:
section_header('Phase 1 — Strukturelles Cleaning')

for year in years_to_split:
    out_clean = INTERIM / f"zh-tram-data-{year}-structural-clean.parquet"

    n_before = lazyframes[year].select(pl.len()).collect().item()
    log(f"\nVor Cleaning: {n_before:,} Zeilen")

    # sink_parquet: Polars Streaming — schreibt in Chunks direkt auf Disk, kein collect()
    structural_cleaning_pipeline(lazyframes[year]).sink_parquet(out_clean)

    n_after = pl.scan_parquet(out_clean).select(pl.len()).collect().item()
    report_step("Gesamt Phase 1", n_before, n_after)
    success(f"Exportiert: {out_clean}")

In [ ]:
# Überblick: gereinigte Jahres-Dateien
clean_files = {
    year: INTERIM / f"zh-tram-data-{year}-structural-clean.parquet"
    for year in years_to_split
}
for year, path in clean_files.items():
    n = pl.scan_parquet(path).select(pl.len()).collect().item()
    log(f"  {year}: {n:,} Zeilen  →  {path.name}")

In [ ]:
# → weiter mit Phase 2: Split aus den jährlichen Dateien

## Split


*Temporal aufteilen — kein Random Shuffle.*

Zeitreihendaten dürfen nicht zufällig gesplittet werden: das Modell würde sonst auf Daten
trainieren die zeitlich nach dem Test liegen — es "kennt die Zukunft".

| Set | Zeitraum | Zeilen (nach Cleaning) |
|:---|:---|:---|
| Train | 2023 + 2024 | → aus Cleaning-Output |
| Test | 2025 | → aus Cleaning-Output |

> Konkretes Verhältnis: siehe Ausgabe unten — berechnet aus den bereinigten Jahresdateien.  
> Test-Daten werden bis zur finalen Evaluation nicht angefasst.

In [ ]:
section_header('Train / Test Split')

# Train = 2023 + 2024 (lazy concat), Test = 2025
lf_train = pl.concat([
    pl.scan_parquet(clean_files["2023"]),
    pl.scan_parquet(clean_files["2024"]),
])
lf_test = pl.scan_parquet(clean_files["2025"])

# Zeilenzahlen (kleine Einzelabfragen, kein collect des vollen Datensatzes)
n_2023  = pl.scan_parquet(clean_files["2023"]).select(pl.len()).collect().item()
n_2024  = pl.scan_parquet(clean_files["2024"]).select(pl.len()).collect().item()
n_test  = pl.scan_parquet(clean_files["2025"]).select(pl.len()).collect().item()
n_train = n_2023 + n_2024
n_total = n_train + n_test

log(f"Train (2023+2024): {n_train:>12,}  ({n_train/n_total*100:.1f}%)")
log(f"  davon 2023:      {n_2023:>12,}")
log(f"  davon 2024:      {n_2024:>12,}")
log(f"Test  (2025):      {n_test:>12,}  ({n_test/n_total*100:.1f}%)")

out_train = INTERIM / "train_raw.parquet"
out_test  = INTERIM / "test_raw.parquet"


lf_train.sink_parquet(out_train)
lf_test.sink_parquet(out_test)

print()
success(f"Split exportiert → {INTERIM}")

## Preprocessing


*Daten modellbereit machen — nach dem Split, Parameter nur aus Train.*

| Schritt | Bedeutung | Status |
|:---|:---|:---|
| **Imputation** | Fehlende Werte füllen | ✅ Meteo Forward/Backward Fill |
| **Scaling** | Wertebereiche normalisieren | → Modeling-Notebook |
| **Encoding** | Kategorien in Zahlen umwandeln | → Modeling-Notebook |
| **Outlier Handling** | Ausreißer behandeln | → Modeling-Notebook |

> Alles was Parameter aus den Daten lernt: erst auf Train fitten, dann auf Test anwenden.

In [ ]:
section_header('Meteo-Imputation ')

#(Forward/Backward Fill)

out_train_prep = PROCESSED / "train_prepared.parquet"
out_test_prep  = PROCESSED / "test_prepared.parquet"

# Nulls vor Imputation (lazy — liest nur Meteo-Spalten)
lf_train_raw = pl.scan_parquet(out_train)
null_check = (
    lf_train_raw.select(METEO_COLS)
    .select([pl.col(c).null_count().alias(c) for c in METEO_COLS])
    .collect()
)
log("Train — Nulls vor Imputation:")
for col in METEO_COLS:
    n = null_check[col][0]
    if n > 0:
        warn(f"  {col:<25} {n:>8,} Nulls")

print()
# impute_meteo_lazy: sort + forward/backward fill, streaming via sink_parquet
log("Starte Imputation auf Train...")
impute_meteo_lazy(pl.scan_parquet(out_train)).sink_parquet(out_train_prep)
log("Starte Imputation auf Test ...")
impute_meteo_lazy(pl.scan_parquet(out_test)).sink_parquet(out_test_prep)

print()
success("Phase 3 abgeschlossen.")
log(f"  {out_train_prep}")
log(f"  {out_test_prep}")

## Feature Engineering


*Neue Spalten aus vorhandenen ableiten — nach dem Split.*

| Kategorie | Features |
|:---|:---|
| Zeit | Stunde · Wochentag · Monat · Saison · Wochenende-Flag · HVZ-Flag |
| Wetter | Regen-Flag · Starkregen-Flag · Wind-Flag · Schnee-Flag · Flut-Flag |
| Event | Feiertag-Flag · Event-Flag · Event-Gewicht |
| Ausfall | `is_canceled` als numerisch |

> Encoding-Parameter (Target-Encoding für Linie, Stadtkreis etc.) werden im Modeling-Notebook auf Train gefittet.

In [ ]:
section_header('Feature Engineering')

# pipe() funktioniert auf LazyFrame genauso wie auf DataFrame

def add_time_features(frame):
    return frame.with_columns([
        pl.col("arrival_schedule").dt.hour().alias("stunde"),
        pl.col("arrival_schedule").dt.weekday().alias("wochentag"),
        pl.col("arrival_schedule").dt.month().alias("monat"),
        (
            pl.when(pl.col("arrival_schedule").dt.month().is_in([12, 1, 2])).then(pl.lit(1))
            .when(pl.col("arrival_schedule").dt.month().is_in([3, 4, 5])).then(pl.lit(2))
            .when(pl.col("arrival_schedule").dt.month().is_in([6, 7, 8])).then(pl.lit(3))
            .otherwise(pl.lit(4))
            .cast(pl.Int8).alias("saison")  # 1=Winter 2=Frühling 3=Sommer 4=Herbst
        ),
        (pl.col("arrival_schedule").dt.weekday() >= 5).alias("ist_wochenende"),
        pl.col("arrival_schedule").dt.hour().is_in([7, 8, 9, 17, 18, 19]).alias("ist_hvz"),
    ])


def add_weather_flags(frame):
    return frame.with_columns([
        (pl.col("precipitation") > 0).alias("hat_regen"),
        (pl.col("precipitation") > 5.0).alias("hat_starkregen"),
        (pl.col("wind_speed") > 40).alias("ist_windig"),            # > 40 km/h
        (
            (pl.col("precipitation") > 0) & (pl.col("temperature") < 2)
        ).alias("hat_schnee"),
        (pl.col("flood_intensity") > 0).alias("hat_flut"),
        pl.col("canceled").cast(pl.Int8).alias("is_canceled"),
    ])


def add_event_features(frame):
    return frame.with_columns([
        (pl.col("event_type") == "Feiertag").alias("ist_feiertag"),
        pl.col("event_name").is_not_null().alias("hat_event"),
        pl.col("event_size").fill_null(0).alias("event_gewicht"),   # 0=kein Event / 1/2/3
    ])

## Export


In [ ]:
section_header('Export')

out_train_feat = PROCESSED / "train_features.parquet"
out_test_feat  = PROCESSED / "test_features.parquet"

(pl.scan_parquet(out_train_prep)
    .pipe(add_time_features)
    .pipe(add_weather_flags)
    .pipe(add_event_features)
    .sink_parquet(out_train_feat))

(pl.scan_parquet(out_test_prep)
    .pipe(add_time_features)
    .pipe(add_weather_flags)
    .pipe(add_event_features)
    .sink_parquet(out_test_feat))

print()
success("Feature Engineering abgeschlossen.")
log(f"  {out_train_feat}")
log(f"  {out_test_feat}")


In [ ]:


lf_train = pl.scan_parquet(out_train_feat)
lf_test = pl.scan_parquet(out_test_feat)

print(lf_train.columns)
print()
print(lf_train.columns)

|           | Train	    | Test      | 
|:---       |:---       |:---       |
| Zeilen    | 60,498,841| 31,719,263| 
| Spalten   | 39        | 39        | 
| Größe	    | 571 MB    | 298 MB    | 


###### 24 originale Spalten + 15 neue Features:

**Zeit:**
* stunde 
* wochentag monat
* saison 
* ist_wochenende 
* ist_hvz

**Wetter:**
* hat_regen 
* hat_starkregen 
* ist_windig 
* hat_schnee 
* hat_flut

**Event:**
* ist_feiertag 
* hat_event 
* event_gewicht

**Ausfall:** 
* is_canceled